In [11]:
using PEPSKit, TensorKit

### Model Parameters ###
L = 3 #Length/width of unit cell
n_0 = round(Int, ((L-1)/2))+1 #Index of the point at the center of the lattice (1-based indexing).
m2 = 1.0 #Bare mass (squared)
m0 = 1 #Basis frequency
l = 0.1 #phi^4 coupling strength
Dim = 5 #Truncated local Hilbert space dimension
d = 2 #Number of spatial dimensions
a = 1.0 #Lattice spacing

### iPEPS Dimensions ###
Dbond = 5
χ = 50

### phi4 Hamiltonian ###
include("phi4_Hamiltonian.jl")
H, φ, φ2, φ4, Π, Π2 = phi4_model(L, m2, m0, l, Dim, d, a)

using JLD2
peps_1 = load_object("Z:/Energy Correlator 2D/VacStates/PEPS,L=1,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2")
env_1 = load_object("Z:/Energy Correlator 2D/VacStates/env,L=1,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2")

CTMRGEnv{TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}, TensorMap{ComplexF64, ComplexSpace, 3, 1, Vector{ComplexF64}}}(TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}[TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}(ComplexF64[-0.2577629711592971 - 0.9595030096118957im, 0.00011899387821582399 + 0.0002449419726071322im, -0.00020273648056223822 - 0.0001062088498565971im, -3.3520623221780915e-5 - 0.00012477796450120995im, 0.000245236278716038 + 1.8086038190909558e-5im, 1.1484402663189845e-5 + 4.27498134615916e-5im, 7.529355218130152e-7 + 2.8027446091837378e-6im, -3.204903021813573e-7 - 1.1930005719338635e-6im, -2.2580538942816222e-6 + 7.348945721773676e-7im, -1.6401657061461704e-7 - 6.105388036009098e-7im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], ℂ^50 ← ℂ^50); TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}(ComplexF64[0.966267251269733 - 0.2

In [12]:
#Convert the 1x1 vacuum tensor to LxL
A1 = peps_1.A[1,1] #1x1 vacuum tensor
AL = fill(A1, (L,L)) #Vacuum tensor copied over an LxL unit cell
peps_L = InfinitePEPS(AL)

InfinitePEPS{TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}}(TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}[TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}(ComplexF64[-0.018670854772324635 + 0.0069097322378792415im, 0.04961534583669015 - 0.012809252225521853im, -0.04648070493327292 - 0.02733860365105849im, -0.0010231848111674374 + 0.0004030161457762554im, 0.006359979578291367 - 0.004210433439053114im, -0.007025730746854253 + 0.02661682678973288im, -0.008360320086303653 + 0.00896601698754955im, 0.003988131105113775 + 0.006786683770065393im, -0.0017205222815733024 + 0.0016019903574357808im, 0.004338620488680373 - 0.005985647739853615im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], ℂ^5 ← (ℂ^5 ⊗ ℂ^5 ⊗ (ℂ^5)' ⊗ (ℂ^5)')) TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}(ComplexF64[-0.018670854772324635 + 0.0069097322378792415im, 0.04961534583669015 

In [13]:
#Convert the 1x1 environment to LxL
env_L = CTMRGEnv(randn, ComplexF64, peps_L, ℂ^χ); #Generate structure of LxL environment

#Replace corner and edge tensors of LxL environment with the 1x1 corner and edge tensors
for r in 1:L, c in 1:L
    for dir in 1:4
        setcorner!(env_L, corner(env_1, dir, 1, 1), dir, r, c)
    end

    for dir in 1:4
        setedge!(env_L, edge(env_1, dir, 1, 1), dir, r, c)
    end
end

In [ ]:
### Save PEPS and CTMRG environment ###
save_object("Z:/Energy Correlator 2D/VacStates/PEPS,L=$L,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2", peps_L)
save_object("Z:/Energy Correlator 2D/VacStates/env,L=$L,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2", env_L)